# Random Forest Classification for TLS Profiling

This notebook evaluates a supervised Random Forest baseline on extracted TLS traffic features. Unlike the anomaly-detection baselines, this model is trained to classify each flow directly into one of three categories: `system`, `malware`, or `unknown`.


In [1]:
import sys
sys.path.append('../../src')


## Feature Groups

- `FLOW`: basic bidirectional flow statistics such as bytes, packets, rates, and duration (`bs`, `ps`, `br`, `pr`, `td`)
- `CTLS`: client-side TLS metadata (`tls.cver`, `tls.ccs`, `tls.cext`, `tls.csg`, `tls.alpn`, `tls.csv`)
- `STLS`: server-side TLS metadata (`tls.sver`, `tls.scs`, `tls.sext`, `tls.ssv`)
- `REC`: ordered sequence of signed TLS record lengths (`tls.rec`, first 20 records)

The notebook evaluates each group individually and all group combinations to understand which feature families are most useful for supervised multiclass classification.


In [2]:
FLOW = ["bs","ps", "br", "pr", "td"]
CTLS = ["tls.cver", "tls.ccs", "tls.cext", "tls.csg", "tls.alpn", "tls.csv"]
STLS = ["tls.sver", "tls.scs", "tls.sext", "tls.ssv"]
REC = ["tls.rec"]


In [3]:
from pathlib import Path

import joblib
import pandas as pd

data_dir = Path("../data-ext")
#data_dir = Path("../data")

print(f"Loading extracted features from {data_dir}.")
df_train = pd.read_parquet(data_dir / "malware_train.parquet")
df_val = pd.read_parquet(data_dir / "malware_val.parquet")
df_test = pd.read_parquet(data_dir / "malware_test.parquet")
df_train_label = pd.read_parquet(data_dir / "malware_train_labels.parquet")
df_val_label = pd.read_parquet(data_dir / "malware_val_labels.parquet")
df_test_label = pd.read_parquet(data_dir / "malware_test_labels.parquet")

preprocessor_path = data_dir / "malware_preprocessor.joblib"
if not preprocessor_path.exists():
    fallback_path = data_dir / "preprocessor.joblib"
    if not fallback_path.exists():
        raise FileNotFoundError(
            f"Could not find {preprocessor_path} or {fallback_path}. Run the data preparation notebook first."
        )
    print(f"{preprocessor_path.name} not found, using {fallback_path.name} instead.")
    preprocessor_path = fallback_path

preprocessor = joblib.load(preprocessor_path)
print(f"Loaded preprocessor from {preprocessor_path}.")


Loading extracted features from ../data-ext.
Loaded preprocessor from ../data-ext/malware_preprocessor.joblib.


In [4]:
from tls_profiling.baselines.model_random_forest import RandomForestBaseline, Config
import numpy as np

from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize

CLASS_NAMES = ["system", "malware", "unknown"]
CLASS_TO_INDEX = {label: idx for idx, label in enumerate(CLASS_NAMES)}
INDEX_TO_CLASS = {idx: label for label, idx in CLASS_TO_INDEX.items()}

RF_CONFIG_CANDIDATES = {
    "balanced_deep": Config(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        max_train_samples=100_000,
    ),
    "balanced_shallow": Config(
        n_estimators=200,
        max_depth=30,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        max_train_samples=100_000,
    ),
}

def encode_labels(series: pd.Series) -> np.ndarray:
    return series.map(CLASS_TO_INDEX).to_numpy(dtype=np.int64)

def decode_labels(values: np.ndarray) -> list[str]:
    return [INDEX_TO_CLASS[int(value)] for value in values]

def select_feature_set(x, feature_set):
    if not feature_set:
        return x

    selected_columns = [
        column for column in x.columns
        if any(column.startswith(prefix) for prefix in feature_set)
    ]
    return x.loc[:, selected_columns]

def compute_multiclass_scores(y_true, y_pred, y_score):
    y_true_bin = label_binarize(y_true, classes=np.arange(len(CLASS_NAMES)))
    per_class_f1 = f1_score(
        y_true,
        y_pred,
        average=None,
        labels=np.arange(len(CLASS_NAMES)),
    )

    return {
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_auc": roc_auc_score(
            y_true,
            y_score,
            labels=np.arange(len(CLASS_NAMES)),
            multi_class="ovr",
            average="macro",
        ),
        "macro_ap": average_precision_score(y_true_bin, y_score, average="macro"),
        "system_f1": per_class_f1[CLASS_TO_INDEX["system"]],
        "malware_f1": per_class_f1[CLASS_TO_INDEX["malware"]],
        "unknown_f1": per_class_f1[CLASS_TO_INDEX["unknown"]],
    }

def choose_best_model(x_train, y_train, x_val, y_val):
    best_name = None
    best_model = None
    best_val_macro_f1 = -np.inf

    for config_name, config in RF_CONFIG_CANDIDATES.items():
        model = RandomForestBaseline(config)
        model.fit(x_train, y_train)

        val_pred = model.predict(x_val)
        val_macro_f1 = f1_score(y_val, val_pred, average="macro")

        if val_macro_f1 > best_val_macro_f1:
            best_name = config_name
            best_model = model
            best_val_macro_f1 = val_macro_f1

    return best_name, best_model, float(best_val_macro_f1)

def evaluate(feature_set):
    y_train = encode_labels(df_train_label["connection_label"])
    y_val = encode_labels(df_val_label["connection_label"])
    y_test = encode_labels(df_test_label["connection_label"])

    x_train_transformed = select_feature_set(preprocessor.transform(df_train), feature_set)
    x_val_transformed = select_feature_set(preprocessor.transform(df_val), feature_set)
    x_test_transformed = select_feature_set(preprocessor.transform(df_test), feature_set)

    best_config_name, model, val_macro_f1 = choose_best_model(
        x_train_transformed,
        y_train,
        x_val_transformed,
        y_val,
    )

    y_pred = model.predict(x_test_transformed)
    y_score = model.predict_proba(x_test_transformed)

    return y_test, y_pred, y_score, val_macro_f1, best_config_name

def debug_csv(feature_set, y_test, y_pred, y_score):
    feature_set_name = "_".join(feature_set)
    output_path = f"tmp/rf_{feature_set_name}.csv"

    pd.DataFrame({
        "y_test": decode_labels(y_test),
        "y_pred": decode_labels(y_pred),
        "p_system": y_score[:, CLASS_TO_INDEX["system"]],
        "p_malware": y_score[:, CLASS_TO_INDEX["malware"]],
        "p_unknown": y_score[:, CLASS_TO_INDEX["unknown"]],
    }).to_csv(output_path, index=False)

def compute_scores(feature_set):
    y_test, y_pred, y_score, val_macro_f1, best_config_name = evaluate(feature_set)
    scores = compute_multiclass_scores(y_test, y_pred, y_score)

    debug_csv(feature_set, y_test, y_pred, y_score)
    return {
        "best_config": best_config_name,
        "val_macro_f1": val_macro_f1,
        **scores,
    }


## Evaluation

This baseline uses the same train, validation, and test splits as the anomaly-detection notebooks, but the task is now multiclass classification rather than one-class profiling.

Protocol:

- `train`: used to fit supervised Random Forest models
- `validation`: used to choose the better of two Random Forest configurations using `macro F1`
- `test`: held out for final reporting only

Target classes:

- `system`
- `malware`
- `unknown`

Reported metrics:

- `MacroF1`: the main score for the multiclass classifier, giving each class equal weight
- `MacroAucScore`: one-vs-rest macro ROC-AUC computed from predicted class probabilities
- `MacroApScore`: one-vs-rest macro average precision computed from predicted class probabilities
- per-class `F1` columns for `system`, `malware`, and `unknown`

Practical notes:

- this is a supervised classifier, so it is not directly comparable to one-class anomaly detectors in terms of training assumptions
- to keep the full feature-ablation study tractable, each Random Forest fit uses a stratified cap of `100,000` training flows


In [5]:
from itertools import combinations

feature_groups = {
    "FLOW": FLOW,
    "CTLS": CTLS,
    "STLS": STLS,
    "REC": REC,
}

rows = []

group_names = list(feature_groups)
for size in range(1, len(group_names) + 1):
    for group_combo in combinations(group_names, size):
        selected_features = []
        for group_name in group_combo:
            selected_features.extend(feature_groups[group_name])

        feature_set_name = ", ".join(group_combo)
        scores = compute_scores(selected_features)

        rows.append({
            "FeatureSet": feature_set_name,
            "BestConfig": scores["best_config"],
            "ValMacroF1": scores["val_macro_f1"],
            "MacroF1": scores["macro_f1"],
            "MacroAucScore": scores["macro_auc"],
            "MacroApScore": scores["macro_ap"],
            "SystemF1": scores["system_f1"],
            "MalwareF1": scores["malware_f1"],
            "UnknownF1": scores["unknown_f1"],
        })

results_df = pd.DataFrame(
    rows,
    columns=[
        "FeatureSet",
        "BestConfig",
        "ValMacroF1",
        "MacroF1",
        "MacroAucScore",
        "MacroApScore",
        "SystemF1",
        "MalwareF1",
        "UnknownF1",
    ],
)
display(results_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
0,FLOW,balanced_deep,0.797507,0.828627,0.952940,0.892222,0.954644,0.899036,0.632201
1,CTLS,balanced_shallow,0.644907,0.636569,0.808068,0.650720,0.886378,0.765490,0.257839
2,STLS,balanced_shallow,0.721822,0.650536,0.922306,0.837112,0.956617,0.590984,0.404007
3,REC,balanced_deep,0.803822,0.813591,0.958074,0.903094,0.965106,0.870685,0.604982
4,"FLOW, CTLS",balanced_deep,0.822130,0.844946,0.956507,0.900089,0.961768,0.909573,0.663497
5,"FLOW, STLS",balanced_deep,0.819910,0.854415,0.960766,0.906056,0.964483,0.917391,0.681372
6,"FLOW, REC",balanced_shallow,0.820754,0.841465,0.964624,0.913005,0.967840,0.897123,0.659434
7,"CTLS, STLS",balanced_shallow,0.744628,0.747115,0.931263,0.855620,0.956824,0.788555,0.495967
8,"CTLS, REC",balanced_deep,0.820518,0.822370,0.956866,0.903877,0.965408,0.880056,0.621648
9,"STLS, REC",balanced_deep,0.804293,0.814651,0.958026,0.903688,0.965182,0.871158,0.607613


## Overall Results

The table below ranks feature sets by `MacroF1`, which is the primary metric for this multiclass baseline. `MacroAucScore` and `MacroApScore` are included to make the supervised baseline easier to compare at a high level with the anomaly-detection notebooks.


In [6]:
overall_df = results_df.sort_values("MacroF1", ascending=False)
display(overall_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.830890,0.862396,0.963645,0.913886,0.965151,0.920537,0.701499
11,"FLOW, CTLS, REC",balanced_deep,0.828647,0.862309,0.963951,0.913546,0.964843,0.921124,0.700959
12,"FLOW, STLS, REC",balanced_deep,0.814008,0.857432,0.962912,0.911110,0.964697,0.916850,0.690751
5,"FLOW, STLS",balanced_deep,0.819910,0.854415,0.960766,0.906056,0.964483,0.917391,0.681372
10,"FLOW, CTLS, STLS",balanced_deep,0.830870,0.851248,0.960773,0.907352,0.964539,0.912404,0.676801
4,"FLOW, CTLS",balanced_deep,0.822130,0.844946,0.956507,0.900089,0.961768,0.909573,0.663497
6,"FLOW, REC",balanced_shallow,0.820754,0.841465,0.964624,0.913005,0.967840,0.897123,0.659434
13,"CTLS, STLS, REC",balanced_deep,0.826988,0.838289,0.958335,0.905582,0.965559,0.895942,0.653365
0,FLOW,balanced_deep,0.797507,0.828627,0.952940,0.892222,0.954644,0.899036,0.632201
8,"CTLS, REC",balanced_deep,0.820518,0.822370,0.956866,0.903877,0.965408,0.880056,0.621648


## System Category

This table isolates how well each feature set classifies the `system` category. A high `SystemF1` means the supervised classifier can separate system traffic cleanly from both malware and residual unknown traffic.


In [7]:
system_df = results_df.sort_values("SystemF1", ascending=False)
display(system_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
6,"FLOW, REC",balanced_shallow,0.820754,0.841465,0.964624,0.913005,0.967840,0.897123,0.659434
13,"CTLS, STLS, REC",balanced_deep,0.826988,0.838289,0.958335,0.905582,0.965559,0.895942,0.653365
8,"CTLS, REC",balanced_deep,0.820518,0.822370,0.956866,0.903877,0.965408,0.880056,0.621648
9,"STLS, REC",balanced_deep,0.804293,0.814651,0.958026,0.903688,0.965182,0.871158,0.607613
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.830890,0.862396,0.963645,0.913886,0.965151,0.920537,0.701499
3,REC,balanced_deep,0.803822,0.813591,0.958074,0.903094,0.965106,0.870685,0.604982
11,"FLOW, CTLS, REC",balanced_deep,0.828647,0.862309,0.963951,0.913546,0.964843,0.921124,0.700959
12,"FLOW, STLS, REC",balanced_deep,0.814008,0.857432,0.962912,0.911110,0.964697,0.916850,0.690751
10,"FLOW, CTLS, STLS",balanced_deep,0.830870,0.851248,0.960773,0.907352,0.964539,0.912404,0.676801
5,"FLOW, STLS",balanced_deep,0.819910,0.854415,0.960766,0.906056,0.964483,0.917391,0.681372


## Malware Category

This section focuses on the `malware` class. Strong `MalwareF1` values indicate that the selected feature family helps the Random Forest carve out malware traffic as a distinct supervised class rather than only separating it from one chosen normal profile.


In [8]:
malware_df = results_df.sort_values("MalwareF1", ascending=False)
display(malware_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
11,"FLOW, CTLS, REC",balanced_deep,0.828647,0.862309,0.963951,0.913546,0.964843,0.921124,0.700959
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.830890,0.862396,0.963645,0.913886,0.965151,0.920537,0.701499
5,"FLOW, STLS",balanced_deep,0.819910,0.854415,0.960766,0.906056,0.964483,0.917391,0.681372
12,"FLOW, STLS, REC",balanced_deep,0.814008,0.857432,0.962912,0.911110,0.964697,0.916850,0.690751
10,"FLOW, CTLS, STLS",balanced_deep,0.830870,0.851248,0.960773,0.907352,0.964539,0.912404,0.676801
4,"FLOW, CTLS",balanced_deep,0.822130,0.844946,0.956507,0.900089,0.961768,0.909573,0.663497
0,FLOW,balanced_deep,0.797507,0.828627,0.952940,0.892222,0.954644,0.899036,0.632201
6,"FLOW, REC",balanced_shallow,0.820754,0.841465,0.964624,0.913005,0.967840,0.897123,0.659434
13,"CTLS, STLS, REC",balanced_deep,0.826988,0.838289,0.958335,0.905582,0.965559,0.895942,0.653365
8,"CTLS, REC",balanced_deep,0.820518,0.822370,0.956866,0.903877,0.965408,0.880056,0.621648


## Unknown Category

The `unknown` label remains heterogeneous, so this table should still be interpreted with care. `UnknownF1` shows how well the classifier treats that residual traffic bucket as its own class instead of collapsing it into `system` or `malware`.


In [9]:
unknown_df = results_df.sort_values("UnknownF1", ascending=False)
display(unknown_df)


,FeatureSet,BestConfig,ValMacroF1,MacroF1,MacroAucScore,MacroApScore,SystemF1,MalwareF1,UnknownF1
14,"FLOW, CTLS, STLS, REC",balanced_deep,0.830890,0.862396,0.963645,0.913886,0.965151,0.920537,0.701499
11,"FLOW, CTLS, REC",balanced_deep,0.828647,0.862309,0.963951,0.913546,0.964843,0.921124,0.700959
12,"FLOW, STLS, REC",balanced_deep,0.814008,0.857432,0.962912,0.911110,0.964697,0.916850,0.690751
5,"FLOW, STLS",balanced_deep,0.819910,0.854415,0.960766,0.906056,0.964483,0.917391,0.681372
10,"FLOW, CTLS, STLS",balanced_deep,0.830870,0.851248,0.960773,0.907352,0.964539,0.912404,0.676801
4,"FLOW, CTLS",balanced_deep,0.822130,0.844946,0.956507,0.900089,0.961768,0.909573,0.663497
6,"FLOW, REC",balanced_shallow,0.820754,0.841465,0.964624,0.913005,0.967840,0.897123,0.659434
13,"CTLS, STLS, REC",balanced_deep,0.826988,0.838289,0.958335,0.905582,0.965559,0.895942,0.653365
0,FLOW,balanced_deep,0.797507,0.828627,0.952940,0.892222,0.954644,0.899036,0.632201
8,"CTLS, REC",balanced_deep,0.820518,0.822370,0.956866,0.903877,0.965408,0.880056,0.621648
